# V3 data audit and Random Forest training
This notebook checks the label/split contract, then calls the same trainer used by the CLI.

In [1]:
# Run once in a clean environment, then restart the kernel.
%pip install -r ../requirements.txt

Note: you may need to restart the kernel to use updated packages.


In [2]:
from pathlib import Path
import sys, json
import pandas as pd
from IPython.display import display
ROOT = Path.cwd().resolve()
if not (ROOT / 'src').is_dir(): ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / 'src'))
DATASET = ROOT / 'data/processed/realworld_v3_features.csv'
frame = pd.read_csv(DATASET)
print('rows:', len(frame), 'columns:', len(frame.columns))
display(pd.crosstab([frame['split'], frame['label_quality']], frame['label']))

rows: 14941 columns: 100


label                               0     1
split       label_quality                  
calibration generated_phishing      0   678
            verified_legitimate   756     0
policy      generated_phishing      0   777
            verified_legitimate   844     0
test        generated_phishing      0   845
            verified_legitimate   935     0
train       generated_phishing      0  4448
            verified_legitimate  4899     0
weak_stress weak_feed               0   759

In [3]:
eligible = frame[frame['training_eligible'].eq(1)].copy()
assert (eligible.groupby('group_id')['split'].nunique() <= 1).all(), 'domain leakage across splits'
assert not frame['sample_id'].duplicated().any(), 'duplicate sample_id'
assert not ((frame['label_quality'] == 'weak_feed') & frame['training_eligible'].eq(1)).any()
assert set(eligible['label_quality']) == {'verified_legitimate', 'generated_phishing'}
print('Data contract passed; eligible rows:', len(eligible))

Data contract passed; eligible rows: 14182


In [4]:
from persianphish_detector.models.train_rf import train
OUTPUT = ROOT / 'artifacts/notebook/detector_v3.joblib'
report = train(DATASET, OUTPUT, n_jobs=-1, search_estimators=700)
display(pd.DataFrame({
    'at_0.5': report['test_at_0_5'],
    'policy_threshold': report['test_at_policy_threshold'],
}).T[['precision','recall','f1','false_positive_rate','roc_auc','pr_auc']])
print('selected:', report['selected_candidate'])
print('artifact:', OUTPUT)

,precision,recall,f1,false_positive_rate,roc_auc,pr_auc
at_0.5,0.976886,0.950296,0.963407,0.020321,0.993334,0.993778
policy_threshold,0.997379,0.900592,0.946517,0.002139,0.993334,0.993778


selected: balanced_general__sigmoid
artifact: Z:\phishing\مجموعه داده\realworld_phishing_detector\artifacts\notebook\detector_v3.joblib
